# 基礎編4
このノートブックでは、トランザクションがアボートするさまざまなパターンの例を示します。

## 準備
実行の準備を行います。

In [1]:
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

## abortTransactionによるアボート
abortTransactionは、現在実行しているトランザクションを強制的に中止します。  
任意の文字列を引数にとることができます。  

スマートコントラクトをデプロイします。

In [2]:
var cid = await deploySmartContract({ a: 'number' }, function basic4(a) {
    if(a==0) {
        abortTransaction("アボート理由はXYZです");
        // ここは実行されません
    }
    return a;
});

引数に0を渡した場合はabortTransactionが実行されアボートします。

In [3]:
var resp = await rpc.call(adminWallet, cid, { a: 0 });
console.log(resp);

{
  txno: 701663,
  txid: 'x3JHFpuyuwvoMaEJe25Nbk9a24ZqTAJw5KpmMsfzyLbpPB',
  status: 'aborted',
  value: 'aborted: アボート理由はXYZです\n    at c026191348:1:23'
}


引数に1を渡した場合はアボートしません。

In [4]:
var resp = await rpc.call(adminWallet, cid, { a: 1 });
console.log(resp);

{
  txno: 701664,
  txid: 'xzYwE2oNqbz8CdbEr887VPRKj6JuPoUUm5uS6BwHkpsGo',
  status: 'ok',
  value: 1
}


## abortTransactionはcatchできない
abortTransactionは、try-catchで囲まれていても動作します。  

スマートコントラクトをデプロイします。

In [5]:
var cid = await deploySmartContract({ a: 'number' }, function basic4(a) {
    try {
        abortTransaction("アボート理由はABCです");
    }catch(err){
        return "caught";
    }
    return a;
});

実行するとcatchされずアボートします。

In [6]:
var resp = await rpc.call(adminWallet, cid);
console.log(resp);

{
  txno: 701669,
  txid: 'xY4QhcEnztUdAMSV9GjGhjEfNV7Gq4Q2TRYMai8qMk23HB',
  status: 'aborted',
  value: 'aborted: アボート理由はABCです\n    at c049661848:1:18'
}


## プログラムの例外でもアボートする
スマートコントラクトの実行中に例外がスローされて、キャッチされなかった場合、トランザクションはアボートします。

スマートコントラクトをデプロイします。

In [7]:
var cid = await deploySmartContract({ a: 'any' }, function basic4(a) {
    var len = a.length;
    return len;
});

引数にaにnullを渡した場合は、例外がthrowされ、トランザクションがアボートします。

In [8]:
var resp = await rpc.call(adminWallet, cid, { a: null });
console.log(resp);

{
  txno: 701674,
  txid: 'x66d6uYRXGsSUty6ZoVCfPyAqhQLGZTsHix4Uf5y94VXx',
  status: 'aborted',
  value: "TypeError: Cannot read property 'length' of null\n    at c015358019:1:13"
}


引数にaに'1234567'を渡した場合は、正常に完了します。

In [9]:
var resp = await rpc.call(adminWallet, cid, { a: '1234567' });
console.log(resp);

{
  txno: 701675,
  txid: 'xHscg88Um6FeYbo6wZYYjB645ryN9qa3Vo6Rofx8TcXy9',
  status: 'ok',
  value: 7
}


## プログラムの例外はcatchできる

スマートコントラクトをデプロイします。

In [10]:
var cid = await deploySmartContract({ a: 'any' }, function basic4(a) {
    try {
        var len = a.length;
    } catch(err){
        return "CAUGHT: " + err;
    }
    return len;
});

引数にaにnullを渡した場合でも、トランザクションのstatusは'ok'となります。

In [11]:
var resp = await rpc.call(adminWallet, cid, { a: null });
console.log(resp);

{
  txno: 701680,
  txid: 'xM7NEY4uZ9jGHQnZZ3zTLrZVb4oqdD7zFf9QmyauwwoKDB',
  status: 'ok',
  value: "CAUGHT: TypeError: Cannot read property 'length' of null"
}


## throw文でアボート
スマートコントラクトの実行中に例外がスローされて、キャッチされなかった場合、トランザクションはアボートします。  
throwされる値によってstatusが変わります。

スマートコントラクトをデプロイします。

In [12]:
var cid = await deploySmartContract({ a: 'number' }, function basic4(a) {
    if(a==0) {
        throw new SyntaxError("msg123");
    }
    if(a==1) {
        throw "msg456";
    }
    if(a==2) {
        throw { x: 789, y: 'ABC' };
    }
    return a;
});

実行します。

In [13]:
var resp = await rpc.call(adminWallet, cid, { a: 0 });
console.log(resp);

{
  txno: 701685,
  txid: 'xPboMtuYmGgGVLjtDNdcPHhnfUaErDxG4trDT4or8TA6LB',
  status: 'aborted',
  value: 'SyntaxError: msg123\n    at c050800358:1:29'
}


In [14]:
var resp = await rpc.call(adminWallet, cid, { a: 1 });
console.log(resp);

{
  txno: 701686,
  txid: 'xHehkQnNtDRVrEkS62JdLJPuqYPuuQu5k6cWTRjBLxTaDB',
  status: 'thrown',
  value: 'msg456'
}


In [15]:
var resp = await rpc.call(adminWallet, cid, { a: 2 });
console.log(resp);

{
  txno: 701687,
  txid: 'xcJGevhUAuUGoPpsnuaYbT86efaDwHXNaTcdHxe3Rc2oz',
  status: 'thrown',
  value: { x: 789, y: 'ABC' }
}


## アボートしたトランザクションは巻き戻される
トランザクションの実行のstatusが'ok'ではない場合、実行中に行われた変更はすべて巻き戻されます。

スマートコントラクトをデプロイします。

In [16]:
var cid = await deploySmartContract({ a: 'number' }, function basic4(a) {
    var x = a + ';' + keyValueGet('x');
    keyValueSet('x', x);
    if(a==0) abortTransaction("ここでアボートします");
    return x;
});

実行します。

In [17]:
var resp = await rpc.call(adminWallet, cid, { a: 2 });
console.log(resp);

{
  txno: 701692,
  txid: 'xMEiTBpzw9Rbf3sZWVHXzS5b6NKDiVzjkaoCrkz36xymt',
  status: 'ok',
  value: '2;null'
}


In [18]:
var resp = await rpc.call(adminWallet, cid, { a: 1 });
console.log(resp);

{
  txno: 701693,
  txid: 'xkikeRAqV6TXttEgjMScWB8aGeVLBHbZi3edbvHvMcLvGB',
  status: 'ok',
  value: '1;2;null'
}


aに0を指定した時には、トランザクションがアボートします。

In [19]:
var resp = await rpc.call(adminWallet, cid, { a: 0 });
console.log(resp);

{
  txno: 701694,
  txid: 'x3EL6J5HQNYPjuG3L2yPpjsAtuHYrYjHFkEQatY8ZXQv5',
  status: 'aborted',
  value: 'aborted: ここでアボートします\n    at c039470301:1:77'
}


In [20]:
var resp = await rpc.call(adminWallet, cid, { a: -1 });
console.log(resp);

{
  txno: 701695,
  txid: 'xohxAVogt5ukRAqA6pfc7CGNmAxTPyA9S7DidDqzyKeuKB',
  status: 'ok',
  value: '-1;1;2;null'
}


In [21]:
var resp = await rpc.call(adminWallet, cid, { a: -2 });
console.log(resp);

{
  txno: 701696,
  txid: 'xe8KbRPoXgWVYst9ig3zYZR8hFPpaab2zbHz8GJ28kdp7',
  status: 'ok',
  value: '-2;-1;1;2;null'
}


アボートしたトランザクションの実行が無かったように記録されています。